# 02 - Clean, profile and mask

**Goal of this phase:** turn `data/raw/reviews_raw.csv` into a safe, analysis-ready file, and record every issue found and every decision made.

**What happens here**
1. Load the raw file and profile it.
2. Run basic quality checks (duplicates, missing values, invalid ratings).
3. **Protect personal details:** contact details (phone, UPI ID, email, @handle) cause the whole review text to be redacted; reference numbers are masked in place.
4. Add helper columns (word count, text tier, repeat count, a flag for reviews worth theme-tagging).
5. Save `data/processed/reviews_clean.csv` and write `docs/data-quality-log.md`.

**Rule for this notebook:** no raw review text is ever printed in the outputs. Only masked text is shown, so this notebook is safe to publish.

**Needs:** `data/raw/reviews_raw.csv` (create it by running `01_collect.ipynb`; the raw file is kept out of GitHub).

## 1. Load and profile

In [ ]:
from pathlib import Path
import re
import pandas as pd

pd.set_option("display.max_colwidth", 160)
pd.set_option("display.width", 140)

def find_project_root():
    here = Path.cwd().resolve()
    for folder in [here, *here.parents]:
        if (folder / "requirements.txt").exists():
            return folder
    raise FileNotFoundError("Could not find requirements.txt. Open the project folder in VS Code first.")

ROOT = find_project_root()
RAW_FILE = ROOT / "data" / "raw" / "reviews_raw.csv"
PROCESSED_DIR = ROOT / "data" / "processed"
DOCS_DIR = ROOT / "docs"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
DOCS_DIR.mkdir(parents=True, exist_ok=True)

if not RAW_FILE.exists():
    raise FileNotFoundError("data/raw/reviews_raw.csv not found. Run 01_collect.ipynb first (the raw file is not stored in GitHub).")

df = pd.read_csv(RAW_FILE, parse_dates=["review_date"])
TOTAL = len(df)
LOG = []      # data-quality log entries are added here as we go
NOTES = []    # observations that are not defects

def log(issue, rows, decision):
    LOG.append({"issue": issue, "rows": int(rows), "pct": round(100 * rows / TOTAL, 1), "decision": decision})

print(f"Loaded data/raw/reviews_raw.csv: {TOTAL:,} rows, {df.shape[1]} columns")

In [ ]:
missing = df.isna().sum()
print("Columns with missing values:")
display(missing[missing > 0].to_frame("missing rows"))
print("Rating values found:", sorted(int(x) for x in df["rating"].unique()))
print("Date range:", df["review_date"].min(), "to", df["review_date"].max())
display(df["app"].value_counts().to_frame("reviews"))

## 2. Basic quality checks

Each check adds a line to the data-quality log with the number of rows affected and what was decided.

In [ ]:
log("Duplicate review_id", df["review_id"].duplicated().sum(),
    "None found. review_id is a safe unique key.")

log("Missing or blank review text", (df["text"].fillna("").str.strip() == "").sum(),
    "None found.")

log("Rating outside 1 to 5", (~df["rating"].between(1, 5)).sum(),
    "None found.")

missing_version = df["app_version"].isna()
log("Missing app_version and review_version", missing_version.sum(),
    "Kept as missing. Version analysis in Phase 5 will use only rows that have a version, and will say so.")

print("Share of reviews with no app version, by app (%):")
display(missing_version.groupby(df["app"]).mean().mul(100).round(1).to_frame("% missing version"))

## 3. Mask personal details

People sometimes paste phone numbers, UPI IDs or transaction numbers into complaints. The raw file stays private, and only protected text is saved to the processed file.

Two levels of protection:
- **Contact details** (email, UPI ID, @handle, phone): the **whole review text** is replaced with `[REDACTED]`. People who paste contact details often sign with their name, and patterns cannot catch names. The rating and date are kept.
- **Reference numbers** (long numbers, PAN-style codes) and links: masked in place with a placeholder, so the complaint text stays usable.

Rules run in order. UPI IDs run before phone numbers, so something like `9876543210@ybl` is caught as one UPI ID.

In [ ]:
# Brand and support handles are not personal, so they are left alone
BRANDS = r"paytm|phonepe|googlepay|gpay|google|cred|bhim|npci|sbi|hdfc|icici|axis|amazon|upi|teambhim\w*"

MASK_RULES = [
    ("EMAIL",       r"[\w.+-]+@[\w-]+\.[\w.-]+",                                     "[EMAIL]"),
    ("URL",         r"https?://\S+|www\.\S+",                                         "[URL]"),
    ("UPI_ID",      r"(?<![\w.])[\w.\-]{2,}@[A-Za-z0-9]{2,}\b",                       "[UPI_ID]"),
    ("HANDLE",      r"(?i)(?<![\w@])@(?!(?:" + BRANDS + r")\b)[a-z0-9_.]{3,}",          "[HANDLE]"),
    ("PHONE",       r"(?<!\d)(?:\+?91[\s-]?)?(?:[6-9]\d{4}[\s-]?\d{5}|[6-9]\d[\s-]\d{8})(?!\d)", "[PHONE]"),
    ("ID_NUMBER",   r"\b[A-Z]{5}\d{4}[A-Z]\b",                                        "[ID_NUMBER]"),
    ("LONG_NUMBER", r"(?<!\d)\d{4}[\s-]\d{4}[\s-]\d{4}(?!\d)|(?<!\d)\d{9,}(?!\d)", "[LONG_NUMBER]"),
]

CONTACT_RULES = {"EMAIL", "UPI_ID", "HANDLE", "PHONE"}   # these identify a person, so the whole review is redacted

def protect(series):
    """Mask reference numbers in place, and redact the whole text if contact details are found."""
    out = series.fillna("").astype(str)
    redact = pd.Series(False, index=out.index)
    counts = {}
    for name, pattern, placeholder in MASK_RULES:
        hit = out.str.contains(pattern, regex=True)
        counts[name] = int(hit.sum())
        if name in CONTACT_RULES:
            redact |= hit
        out = out.str.replace(pattern, placeholder, regex=True)
    return out.mask(redact, "[REDACTED]"), redact, counts

df["text_masked"], redacted, mask_counts = protect(df["text"])
df["pii_masked"] = df["text_masked"].ne(df["text"].fillna("").astype(str))

print(f"Reviews changed by protection: {int(df['pii_masked'].sum()):,} of {TOTAL:,}")
print(f"  of which fully redacted (contact details found): {int(redacted.sum()):,}")
display(pd.Series(mask_counts, name="reviews hit").to_frame())

In [ ]:
# Demonstration on MADE-UP text (real reviews are never printed here, so this notebook is safe to publish)
demo = [
    "Call me on 98765 43210 about my refund",
    "Money went to demo.user@okaxis but never arrived",
    "Mail me at sample.user@example.com or see https://example.com/help",
    "Reference 123456789012 still pending, please check",
    "Please help @someuser, and thanks @Paytm",
    "I paid Rs 5,000 on 12-10-2026 and the app version is 4.0.29",
]
demo_out, _, _ = protect(pd.Series(demo))
display(pd.DataFrame({"made-up input": demo, "stored as": demo_out}))

In [ ]:
for name, count in mask_counts.items():
    if name in CONTACT_RULES:
        decision = "Detected. The whole review text is replaced with [REDACTED]; rating and date are kept."
    else:
        decision = "Masked in place with a placeholder. The raw text is not saved to the processed file."
    log(f"Personal detail found: {name}", count, decision)

log("Reviews fully redacted (contact details found)", redacted.sum(),
    "Text removed because names cannot be detected by patterns. Rating and date kept for rating analysis; excluded from theme tagging.")

# Safety check after protection. Dates are removed first so they are not mistaken for phone numbers.
no_dates = df["text_masked"].str.replace(r"\d{1,2}[-/.]\d{1,2}[-/.]\d{2,4}", " ", regex=True)
residual_digits = int(no_dates.str.contains(r"(?<!\d)(?:\d[\s.\-]?){10,}(?!\d)", regex=True).sum())
residual_at = int(df["text_masked"].str.contains("@", regex=False).sum())
print(f"Reviews still containing a run of 10+ digits (spaces or hyphens allowed): {residual_digits}")
print(f"Reviews still containing an '@' : {residual_at}")

log("Still contains a run of 10+ digits after protection", residual_digits,
    "Counts only, not printed. Spot-checked when this notebook was built: ticket numbers and gibberish.")
log("Still contains an '@' after protection", residual_at,
    "Counts only, not printed. Spot-checked when this notebook was built: brand mentions, times and gibberish. Names inside other reviews cannot be caught, so the README says anonymity is not guaranteed.")

## 4. Text profile and helper columns

Most reviews are very short ("good", "nice app"). They count fully for ratings and volumes, but carry little complaint text.

In [ ]:
t = df["text_masked"]
df["word_count"] = t.str.split().str.len()
df["has_letters"] = t.str.contains(r"[^\W\d_]", regex=True)            # False for emoji/symbol-only reviews
df["has_devanagari"] = t.str.contains(r"[\u0900-\u097F]", regex=True)   # Hindi script
df["text_tier"] = pd.cut(df["word_count"], bins=[-1, 2, 5, 10**6],
                         labels=["very_short", "short", "detailed"]).astype(str)

# Normalised text used only to find repeats (lowercase, punctuation removed, spaces collapsed).
# Emoji-only reviews are left out, otherwise they would all count as repeats of each other.
df["_key"] = t.str.lower().str.replace(r"[^\w\s]", " ", regex=True).str.split().str.join(" ")
df["text_repeat_count"] = 1
has_text = df["has_letters"]
df.loc[has_text, "text_repeat_count"] = df[has_text].groupby(["app", "_key"])["review_id"].transform("size")

print("Reviews by text tier (very_short = 2 words or fewer, short = 3 to 5, detailed = 6 or more):")
display(df["text_tier"].value_counts().reindex(["very_short", "short", "detailed"]).to_frame("reviews"))
print("Detailed reviews as % of each app's reviews:")
display((df["text_tier"].eq("detailed").groupby(df["app"]).mean() * 100).round(1).to_frame("% detailed"))

In [ ]:
log("Very short text (2 words or fewer)", (df["word_count"] <= 2).sum(),
    "Kept for rating and volume analysis. Not sent for theme tagging.")

log("Emoji or symbol-only text (no letters)", (~df["has_letters"]).sum(),
    "Kept for ratings. Flagged has_letters = False and excluded from tagging.")

log("Hindi (Devanagari) script", df["has_devanagari"].sum(),
    "Kept and flagged. The LLM step can read it, but volume is small.")

extra_copies = df[df["has_letters"]].duplicated(["app", "_key"]).sum()
log("Review text repeats another review in the same app (extra copies)", extra_copies,
    "Kept, not deleted. Short phrases like 'good' are repeated by real users. Flagged with text_repeat_count.")

long_repeats = df[(df["word_count"] >= 6) & df["has_letters"] & (df["text_repeat_count"] >= 3)]
log("Possible copy-paste or spam (6+ words, repeated 3+ times)", len(long_repeats),
    "Kept and flagged through text_repeat_count. Reviewed again in Phase 5.")

print("Most repeated long texts (masked):")
display(long_repeats.groupby(["app", "_key"]).size().sort_values(ascending=False).head(5).to_frame("copies"))

## 5. Which reviews go to theme tagging?

Rule (decision #12 in the decision log): a review is a **tagging candidate** if it contains letters **and** either
- has 6 or more words, or
- has a rating of 1 or 2 with at least 3 words (short complaints such as "payment failed again").

Everything else stays in the dataset for rating and volume analysis only.

In [ ]:
df["tag_candidate"] = df["has_letters"] & (
    (df["word_count"] >= 6) | ((df["rating"] <= 2) & (df["word_count"] >= 3))
)

log("Reviews selected for theme tagging (tag_candidate)", df["tag_candidate"].sum(),
    "Flagged. Phase 3 tags only these, and removes repeated texts before tagging to save compute.")

by_app = df.groupby("app").agg(all_reviews=("review_id", "size"), tag_candidates=("tag_candidate", "sum"))
by_app["% of app"] = (100 * by_app["tag_candidates"] / by_app["all_reviews"]).round(1)
display(by_app)

## 6. Observations that are not defects

These don't need fixing, but they affect how the analysis must be read.

In [ ]:
# 1) How often does each app's developer reply?
reply = (df.groupby("app")["has_dev_reply"].mean() * 100).round(1)
display(reply.to_frame("% reviews with a developer reply"))
NOTES.append("Developer reply rates differ widely by app (" +
             ", ".join(f"{a} {v}%" for a, v in reply.sort_values(ascending=False).items()) +
             "). Ratings can change after a reply, so comparisons between apps need this caveat.")

# 2) Days when the share of 1-star reviews was more than double the app's average (days with 100+ reviews)
df["review_day"] = df["review_date"].dt.date
daily = df.groupby(["app", "review_day"]).agg(
    reviews=("rating", "size"), one_star_share=("rating", lambda s: (s == 1).mean())).reset_index()
overall = df.groupby("app")["rating"].apply(lambda s: (s == 1).mean()).rename("app_avg_one_star_share")
daily = daily.join(overall, on="app")
spikes = daily[(daily["reviews"] >= 100) & (daily["one_star_share"] > 2 * daily["app_avg_one_star_share"])]
display(spikes.round(3))
for _, r in spikes.iterrows():
    NOTES.append(f"{r['app']} on {r['review_day']}: {r['one_star_share']:.0%} of {int(r['reviews'])} reviews were 1-star "
                 f"(app average {r['app_avg_one_star_share']:.0%}). To investigate in Phase 5. Cause not yet known.")

NOTES.append("The first collection run capped each app at 5,000 reviews, which gave windows from 10 to 151 days. "
             "It was replaced by a fixed 60-day window for every app (see the collection note).")

## 7. Save the processed file and the data-quality log

In [ ]:
cols = ["app", "review_id", "rating", "text_masked", "thumbs_up", "review_version", "app_version",
        "review_date", "review_day", "has_dev_reply", "word_count", "text_tier", "has_letters",
        "has_devanagari", "pii_masked", "text_repeat_count", "tag_candidate", "collected_on"]
clean = df[cols].copy()

# Data minimisation: some people type their own name as the whole review, and patterns cannot detect names.
# So the processed (public) file keeps review text only where theme analysis needs it.
# Every review keeps its rating, date, version and helper columns for the rating analysis.
n_not_stored = int((~clean["tag_candidate"]).sum())
clean["text_masked"] = clean["text_masked"].where(clean["tag_candidate"], "[not stored]")
log("Review text not stored in the processed file (not needed for theme analysis)", n_not_stored,
    "Replaced with [not stored]. Reason: some reviews are just a person's name, which patterns cannot detect. Ratings, dates and helper columns are kept for these reviews.")
clean["review_day"] = clean["review_day"].astype(str)
clean.to_csv(PROCESSED_DIR / "reviews_clean.csv", index=False)
size_mb = (PROCESSED_DIR / "reviews_clean.csv").stat().st_size / 1e6
print(f"Saved data/processed/reviews_clean.csv: {len(clean):,} rows, {size_mb:.1f} MB (raw text not included; text kept only for tagging candidates)")

# ---- Write the data-quality log (markdown) ----
lines = ["# Data-quality log", "",
         f"Source file: `data/raw/reviews_raw.csv` ({TOTAL:,} reviews). Numbers below are produced by `notebooks/02_clean_and_mask.ipynb`.", "",
         "| # | Issue | Rows affected | % of data | Decision |", "|---|---|---|---|---|"]
for i, e in enumerate(LOG, 1):
    lines.append(f"| {i} | {e['issue']} | {e['rows']:,} | {e['pct']}% | {e['decision']} |")
lines += ["", "## Observations (not defects)", ""]
lines += [f"- {n}" for n in NOTES]
(DOCS_DIR / "data-quality-log.md").write_text("\n".join(lines) + "\n", encoding="utf-8")
print("Saved docs/data-quality-log.md with", len(LOG), "entries")

## Next

- Read `docs/data-quality-log.md` and check that each number makes sense to you.
- **Phase 3:** theme tagging on the `tag_candidate` reviews (runs in Colab for the free GPU).